In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import timm
import os
import sys
import gc
import matplotlib.pyplot as plt

# Set absolute paths
PROJECT_ROOT = r'd:\Projects\ML_Algorithms\batchsize'
IMAGE_PATH = r'E:\covidx'
TRAIN_CSV = os.path.join(IMAGE_PATH, 'covidx_merged.csv')

# Add project root and functions to path
if PROJECT_ROOT not in sys.path: sys.path.append(PROJECT_ROOT)
functions_path = os.path.join(PROJECT_ROOT, "functions")
if functions_path not in sys.path: sys.path.append(functions_path)

from dataset import COVIDCXNetDataset
from train import train
from evaluation import plot_results

# Fixed Seed for Reproducibility
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


c:\Users\Furkan\Miniconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
batch_sizes = [8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192]
num_epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


Using device: cuda


In [3]:
results = {}

for bs in batch_sizes:
    print(f"\n{'='*30}\nRunning experiment with Batch Size: {bs}\n{'='*30}")
    
    try:
        # Dataset & Dataloader
        ROOT_DATA_DIR = os.path.dirname(IMAGE_PATH) # E:\\
        train_ds = COVIDCXNetDataset(TRAIN_CSV, ROOT_DATA_DIR, transform=transform, split='train')
        val_ds = COVIDCXNetDataset(TRAIN_CSV, ROOT_DATA_DIR, transform=transform, split='val')
        
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True, num_workers=4, pin_memory=True)
        val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False, num_workers=4, pin_memory=True)
        
        model = timm.create_model('resnet50', pretrained=True)
        model.reset_classifier(num_classes=2)
        model.to(device)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        
        # Prepare paths
        save_dir = os.path.join(PROJECT_ROOT, 'models')
        log_dir = os.path.join(PROJECT_ROOT, 'logs')
        os.makedirs(save_dir, exist_ok=True)
        os.makedirs(log_dir, exist_ok=True)

        save_path = os.path.join(save_dir, f'resnet50_bs{bs}.pth')
        log_path = os.path.join(log_dir, f'resnet50_bs{bs}.log')

        # Train
        train_losses, train_accs, val_losses, val_accs = train(
            model, train_loader, val_loader, criterion, optimizer, device,
            save_path=save_path, num_epochs=num_epochs, patience=3, log_path=log_path
        )

        # Store results
        results[bs] = max(val_accs)
        print(f"Best Val Acc for BS {bs}: {results[bs]:.4f}")
        
    except torch.cuda.OutOfMemoryError:
        print(f"CRITICAL: Out of Memory for Batch Size {bs}. Skipping higher batch sizes.")
        break
    except Exception as e:
        print(f"An error occurred with Batch Size {bs}: {e}")
    finally:
        # Cleanup memory
        if 'model' in locals(): del model
        if 'optimizer' in locals(): del optimizer
        if 'train_loader' in locals(): del train_loader
        if 'val_loader' in locals(): del val_loader
        if 'train_ds' in locals(): del train_ds
        if 'val_ds' in locals(): del val_ds
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print(f"Memory cleared after BS {bs}.")


In [ ]:
# Visualization
if results:
    plt.figure(figsize=(10, 6))
    plt.plot(list(results.keys()), list(results.values()), marker='o', linestyle='-', color='b')
    plt.title('ResNet50: Batch Size vs Validation Accuracy')
    plt.xlabel('Batch Size')
    plt.ylabel('Best Validation Accuracy')
    plt.grid(True)
    plt.xticks(list(results.keys()))
    plt.show()
else:
    print("No successful experiments to visualize.")
